In [45]:
import pandas as pd
import numpy as np

from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, recall_score, roc_curve, roc_auc_score, accuracy_score, recall_score, auc
from xgboost import XGBClassifier

In [11]:
model_data = pd.read_csv("../data/recoded_genotypes.csv")
model_data.head()

,AMOSTRA,ETNIA,SEXO,IDADE,ELISA_IgG,VARIANTE,MODELO,CODIGO
0,1,1.0,1,71.0,0,IL17_rs2275913G_A,Additive,0.0
1,1,1.0,1,71.0,0,IL17_rs2275913G_A,Dominant,0.0
2,1,1.0,1,71.0,0,IL17_rs2275913G_A,Recessive,0.0
3,2,1.0,1,69.0,1,IL17_rs2275913G_A,Additive,0.0
4,2,1.0,1,69.0,1,IL17_rs2275913G_A,Dominant,0.0


In [43]:
cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=20030407
)

scoring = {
    "auc": "roc_auc",
    "accuracy": "accuracy",
    "specificity": make_scorer(recall_score, pos_label=0)
}

models = {
    "LR": LogisticRegression(max_iter=5000),
    "RF": RandomForestClassifier(n_estimators=500, random_state=20030407),
    "XGBoost": XGBClassifier(random_state=20030407, eval_metric="logloss")
}

In [49]:
roc_rows = []
metric_rows = []

genetic_models = model_data["MODELO"].unique()

for genetic_model in genetic_models:

    print(f"Executando modelo genético: {genetic_model}")

    df_model = (
        model_data
        .loc[model_data["MODELO"] == genetic_model]
        .pivot(
            index=["AMOSTRA", "ETNIA", "SEXO", "IDADE", "ELISA_IgG"],
            columns="VARIANTE",
            values="CODIGO"
        )
        .reset_index()
    )

    genotype_cols = [c for c in df_model.columns if "rs" in c]

    X = df_model[["ETNIA", "SEXO", "IDADE"] + genotype_cols]
    y = df_model["ELISA_IgG"]

    for classifier_name, estimator in models.items():

        print(f"  {classifier_name}")

        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("classifier", estimator)
        ])

        for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):

            X_train = X.iloc[train_idx]
            X_test = X.iloc[test_idx]

            y_train = y.iloc[train_idx]
            y_test = y.iloc[test_idx]

            pipe.fit(X_train, y_train)

            y_prob = pipe.predict_proba(X_test)[:, 1]
            y_pred = pipe.predict(X_test)

            fpr, tpr, thresholds = roc_curve(y_test, y_prob)

            ## salva todos os pontos da curva ROC
            roc_rows.extend([
                {
                    "GeneticModel": genetic_model,
                    "Classifier": classifier_name,
                    "Fold": fold,
                    "FPR": fp,
                    "TPR": tp
                }
                for fp, tp in zip(fpr, tpr)
            ])

            ## salva métricas do fold
            metric_rows.append({
                "GeneticModel": genetic_model,
                "Classifier": classifier_name,
                "Fold": fold,
                "AUC": roc_auc_score(y_test, y_prob),
                "Accuracy": accuracy_score(y_test, y_pred),
                "Specificity": recall_score(
                    y_test,
                    y_pred,
                    pos_label=0
                )
            })

roc_curve_data = pd.DataFrame(roc_rows)
roc_curve_data.to_csv("../data/roc_curve_all_snps.csv", index=False)

models_evaluation = pd.DataFrame(metric_rows)
models_evaluation.to_csv("../data/models_stats_all_snps.csv", index=False)

Executando modelo genético: Additive
  LR
  RF
  XGBoost
Executando modelo genético: Dominant
  LR
  RF
  XGBoost
Executando modelo genético: Recessive
  LR
  RF
  XGBoost
